In [2]:
# 기본 설정 및 라이브러리 불러오기
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain import hub
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain.tools.retriever import create_retriever_tool
from langgraph.prebuilt import tools_condition, ToolNode
from langgraph.graph.message import add_messages
from langgraph.graph import END, StateGraph, START
from typing import Annotated, Literal, Sequence, TypedDict
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# 에이전트 상태 정의
# 금융 정보를 제공하는 웹페이지를 크롤링 -> 텍스트 분할 -> 벡터스토어에 저장

# 크롤링할 웹페이지 URL 목록
urls = ["https://finance.naver.com/","https://finance.yahoo.com/","https://finance.daum.net/",]

# 각 URL에서 문서 로드
docs = [WebBaseLoader(url).load() for url in urls]
docs

[[Document(metadata={'source': 'https://finance.naver.com/', 'title': '네이버페이 증권', 'language': 'ko'}, page_content='\n\n\n네이버페이 증권\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n메인 메뉴로 바로가기\n본문으로 바로가기\n\n\n\n\n\n\n\n\n\n\n네이버\n\n\n\n\n\n페이\n\n\n\n\n\n\n증권\n\n\n\n\n\n\n증권 종목명·지수명 검색\n\n\n\n\n\n\n\n검색\n\n\n자동완성\n\n\n\n\n\n\n\n@code@\n@txt@\n@market@\n\n@full_txt@\n@in_code@\n@in_name@\n@in_link@\n@in_market@\n\n\n\n\n\t\t\t\t\t\t\t\t\t\t\t\t공모주와 해외 종목은 모바일 페이지로 이동합니다.\n\t\t\t\t\t\t\t\t\t\t\t\n\n\n\n\n\n\n\n\n\t\t\t\t\t\t\t\t\t\t\t\t현재 자동완성 기능을 사용하고 계십니다.\n\t\t\t\t\t\t\t\t\t\t\t\n\n\n\n\n\n\n\n\n\n\t\t\t\t\t\t\t\t\t\t\t\t자동완성 기능이 활성화되었습니다.\n\t\t\t\t\t\t\t\t\t\t\t\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n증권 홈선택됨\n국내증시\n해외증시\n시장지표\n리서치\n뉴스\nMY\n\n\n\n\n\n\n\n\n\n본문시작\n\n오늘의 코스피/코스닥 지수\n2025년 08월 26일 장마감\n코스피 지수 3,179.36 전일대비 하락 30.50 마이너스 0.95 퍼센트\n코스닥 지수 801.66 전일대비 상승 3.64 플러스 0.46 퍼센트\n\n\n\n\n\n최근조회종목\nMY STOCK\n\n\n\n최근 조회종목 리스트\n\n\n\n\n\n\n\n\n\n\n\n\n\n주요뉴스\n\n\n“외국인 자금

In [4]:
docs_list = [item for sublist in docs for item in sublist] 
# item을 docs_list에 넣겠다 

# for문을 풀어서 쓰면
# docs_list = []
# for sublist in docs:
#     for item in sublist:
#         docs_list.append(item)

In [5]:
# 문서 분할
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=100,
    chunk_overlap=50
)

doc_splits = text_splitter.split_documents(docs_list)

In [6]:
# 벡터스토어에 문서 추가
vectorstore = Chroma.from_documents(
    documents=doc_splits,   # 텍스트 분할
    collection_name='rag-chroma',   # 이름표 같은 역할
    embedding=OpenAIEmbeddings(),
)

# 검색기 역할을 하는 retriever
retriever = vectorstore.as_retriever()

In [7]:
# 에이전트의 상태를 나타내는 데이터 구조 정의
# AgentState 클래스를 정의
# 메시지 기본값은 대체해주고 add_messages 는 업데이트가 어떻게 처리되어야하는지 정의
class AgentState(TypedDict): # TypedDict를 물려받겠다
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [8]:
# 벡터스토어에서 검색을 가능하게 하는 도구 생성
retriever_tool = create_retriever_tool(
    retriever, # 이전에 생성한 retriever 사용해 문서 검색
    "retriever_blog_posts",  # 도구의 이름
    "네이버, 야후, 다음의 금융 관련 정보를 검색하고 반환합니다.", # 설명
)
tools = [retriever_tool] 

In [ ]:
# 검색된 문서가 사용자 질문과 관련이 있는지 평가하는 grade_documents 함수 정의
# generate : 관련성 있으면 반환, 답변 생성으로 이동
# rewrite : 관련성 없으면 반환, 질문 재작성으로 이동

def grade_documents(state) -> Literal['generate', 'rewrite']: 
    """ 
    검색된 문서가 질문과 관련이 있는지 평가합니다.
    
    Args:
        state(message): 현재 상태
    Returns: 
        str: 문서의 관련성에 따라 다음 노드 결정('generate' 또는 'rewrite')
    """
    print("-----문서 관련성 평가-----")
    
    # 함수 안 정의 : 함수 내부에서만 사용, 외부와 독립 -> 코드 캡슐화/가독성높음, 다른 함수에서 재사용불가 -> 중복 정의 필요
    # 데이터 모델 정의 -> yes or no
    class grade(BaseModel):
        """ 관련성 평가를 위한 이진 점수(binary score) """
        binary_score: str = Field(description="관련성 점수 'yes' 또는 'no'")  # field라는 자료형

    # LLM 모델 정의
    model = ChatOpenAI(temperature=0, model='gpt-4o-mini', streaming=True) # streaming=True : 토큰 단위로 조금씩 흘려받는 방식

    # LLM에 데이터 모델 적용(grade를 적용한 최종모델)
    llm_with_tool = model.with_structured_output(grade)

    prompt = PromptTemplate(
        template="""당신은 사용자 질문에 대한 검색된 문서의 관련성을 평가하는 평가자입니다.\n
        여기 검색된 문서가 있습니다:\n\n{context}\n\n
        여기 사용자 질문이 있습니다: {question}\n
        문서가 사용자 질문과 관련된 키워드 또는 의미를 포함하면 관련성이 있다고 평가하세요.\n
        문서가 질문과 관련이 있는지 여부를 나타내기 위해 'yes' 또는 'no'로 이진 점수를 주세요.""",
        input_variables=["context", "question"],
    )

    # 체인 생성
    chain = prompt | llm_with_tool

